# Notebook 1: Non-IB Customer — Data Preparation & Feature Engineering (Y-Shape Pipeline)

Notebook này thực hiện toàn bộ luồng chuẩn bị dữ liệu:
1. **Master Prep**: Khôi phục giá trị khuyết thiếu (Missing Values) bằng Median/Mode, và áp dụng thuật toán Capping 1%-99% cho Outlier.
2. **Feature Engineering**: Tính toán các feature cốt lõi (Demographics, Deposit, Lending, Ratios, Trends).


In [1]:
# ============================================================
# IMPORTS & CONFIGURATION
# ============================================================
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 30)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == 'Cluster_nonIB':
    PROJECT_DIR = PROJECT_DIR.parent

CLUSTER_DIR = PROJECT_DIR / 'Cluster_nonIB'
DATA_DIR = PROJECT_DIR / 'cleaned_data'
OUTPUT_DIR = CLUSTER_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REFERENCE_DATE = pd.Timestamp('2020-01-01')

print(f'Cleaned data dir: {DATA_DIR.resolve()}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')

Cleaned data dir: C:\Users\Admin\Desktop\gcon\cleaned_data
Output dir: C:\Users\Admin\Desktop\gcon\Cluster_nonIB\output


In [2]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================
def read_clean_table(name, **csv_kwargs):
    parquet_path = DATA_DIR / f'{name}.parquet'
    csv_path = DATA_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path, **csv_kwargs)
    raise FileNotFoundError(f'Cannot find {parquet_path} or {csv_path}')

def compute_slope(series):
    y = series.dropna().values
    if len(y) < 2: return 0.0
    x = np.arange(len(y))
    slope, _, _, _, _ = stats.linregress(x, y)
    return slope

def cap_outliers(df, cols, lower_percentile=0.01, upper_percentile=0.99):
    df_capped = df.copy()
    for col in cols:
        if col in df_capped.columns:
            lower = df_capped[col].quantile(lower_percentile)
            upper = df_capped[col].quantile(upper_percentile)
            df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
            print(f"Capped {col} at [{lower:.2f}, {upper:.2f}]")
    return df_capped

def save_output(df, name):
    parquet_path = OUTPUT_DIR / f'{name}.parquet'
    csv_path = OUTPUT_DIR / f'{name}.csv'
    df.to_parquet(parquet_path, index=False)
    # df.to_csv(csv_path, index=False) # Uncomment if CSV is needed
    print(f'Saved {name}: shape={df.shape}')

## 1. Master Prep & Load Non-IB Customers

In [3]:
# Load customer data
df_customer = read_clean_table('customer_clean', low_memory=False)

# Master Prep: Impute Missing Age/Sex
df_customer['CLIENT_SEX'] = df_customer['CLIENT_SEX'].fillna(df_customer['CLIENT_SEX'].mode()[0])
df_customer['DATE_OF_BIRTH'] = pd.to_datetime(df_customer['DATE_OF_BIRTH'], errors='coerce')
df_customer['AGE'] = (REFERENCE_DATE - df_customer['DATE_OF_BIRTH']).dt.days / 365.25
df_customer.loc[df_customer['AGE'] < 18, 'AGE'] = np.nan
df_customer.loc[df_customer['AGE'] > 100, 'AGE'] = np.nan
df_customer['AGE'] = df_customer['AGE'].fillna(df_customer['AGE'].median())

# Tenure
df_customer['CLIENT_CREATE_DATE'] = pd.to_datetime(df_customer['CLIENT_CREATE_DATE'], errors='coerce')
df_customer['TENURE_DAYS'] = (REFERENCE_DATE - df_customer['CLIENT_CREATE_DATE']).dt.days
df_customer['TENURE_DAYS'] = df_customer['TENURE_DAYS'].fillna(0).clip(lower=0)

# Filter Non-IB: IB_REGISTER_DATE is NaN
df_nonib = df_customer[df_customer['IB_REGISTER_DATE'].isna()].copy()
print(f'Total customers: {len(df_customer):,}')
print(f'Non-IB customers: {len(df_nonib):,} ({len(df_nonib)/len(df_customer)*100:.1f}%)')

NONIB_IDS = set(df_nonib['CUSTOMER_NUMBER'].unique())
display(df_nonib.head())

Total customers: 285,929
Non-IB customers: 127,000 (44.4%)


,CUSTOMER_NUMBER,CLIENT_SEX,CLIENT_CREATE_DATE,DATE_OF_BIRTH,STAFF,IB_REGISTER_DATE,EB_REGISTER_CHANNEL,SMS,VERIFY_METHOD,AGE,Occupation_Group,Education_Level,Marital_Status,TENURE_DAYS
158929,62117,M,2019-02-26,1975-12-19,N,NaT,None,None,None,44.035592,Commercial associate,higher education,Married,309
158930,900734,F,2019-12-26,1983-06-08,N,NaT,None,None,None,36.566735,Businessman,higher education,Married,6
158931,248998,M,2019-11-22,1981-09-27,N,NaT,None,None,None,38.261465,Commercial associate,secondary/ secondary special,Married,40
158932,954356,M,2019-03-15,1998-09-09,N,NaT,None,None,None,21.311431,Commercial associate,higher education,Single / not married,292
158933,671766,M,2019-11-07,1985-07-02,N,NaT,None,None,None,34.499658,Commercial associate,secondary/ secondary special,Single / not married,55


## 2. Demographics Feature Engineering

In [4]:
# Age bins
bins = [0, 25, 35, 45, 55, 120]
labels = ['<25', '25-35', '35-45', '45-55', '55+']
df_nonib['AGE_GROUP'] = pd.cut(df_nonib['AGE'], bins=bins, labels=labels)

# Gender
df_nonib['CLIENT_SEX'] = df_nonib['CLIENT_SEX'].fillna('UNKNOWN')
gender_dummies = pd.get_dummies(df_nonib['CLIENT_SEX'], prefix='SEX', dtype='int8')
df_nonib = pd.concat([df_nonib, gender_dummies], axis=1)

demo_cols = ['CUSTOMER_NUMBER', 'AGE', 'AGE_GROUP', 'TENURE_DAYS', 'CLIENT_SEX'] + \
            [c for c in df_nonib.columns if c.startswith('SEX_')]
df_demo = df_nonib[demo_cols].copy()

print(f'Demographics features: {len(demo_cols)} columns')
display(df_demo.head())

Demographics features: 7 columns


,CUSTOMER_NUMBER,AGE,AGE_GROUP,TENURE_DAYS,CLIENT_SEX,SEX_F,SEX_M
158929,62117,44.035592,35-45,309,M,0,1
158930,900734,36.566735,35-45,6,F,1,0
158931,248998,38.261465,35-45,40,M,0,1
158932,954356,21.311431,<25,292,M,0,1
158933,671766,34.499658,25-35,55,M,0,1


## 3. Deposit Features

In [5]:
df_deposit = read_clean_table('deposit_clean')
# Master Prep: Cap Outliers
df_deposit = cap_outliers(df_deposit, ['AVG_CA_BALANCE', 'AVG_TD_BALANCE'])

dep_nonib = df_deposit[df_deposit['CUSTOMER_NUMBER'].isin(NONIB_IDS)].copy()
dep_nonib['MONTH'] = pd.to_datetime(dep_nonib['MONTH'], errors='coerce')
dep_nonib = dep_nonib.sort_values(['CUSTOMER_NUMBER', 'MONTH'])

dep_agg = dep_nonib.groupby('CUSTOMER_NUMBER').agg(
    CA_BALANCE_MEAN   = ('AVG_CA_BALANCE', 'mean'),
    CA_BALANCE_MAX    = ('AVG_CA_BALANCE', 'max'),
    CA_BALANCE_MIN    = ('AVG_CA_BALANCE', 'min'),
    CA_BALANCE_LAST   = ('AVG_CA_BALANCE', 'last'),
    CA_BALANCE_STD    = ('AVG_CA_BALANCE', 'std'),
    TD_BALANCE_MEAN   = ('AVG_TD_BALANCE', 'mean'),
    TD_BALANCE_MAX    = ('AVG_TD_BALANCE', 'max'),
    TD_BALANCE_LAST   = ('AVG_TD_BALANCE', 'last'),
    TD_BALANCE_STD    = ('AVG_TD_BALANCE', 'std'),
    CA_ACCT_MAX       = ('COUNT_CA_ACCT', 'max'),
    TD_ACCT_MAX       = ('COUNT_TD_ACCT', 'max'),
    MONTHS_WITH_DEPOSIT = ('MONTH', 'nunique'),
).reset_index()

dep_agg['CA_BALANCE_STD'] = dep_agg['CA_BALANCE_STD'].fillna(0)
dep_agg['TD_BALANCE_STD'] = dep_agg['TD_BALANCE_STD'].fillna(0)
display(dep_agg.head())

Capped AVG_CA_BALANCE at [0.00, 68224780.79]
Capped AVG_TD_BALANCE at [0.00, 1000000000.00]


,CUSTOMER_NUMBER,CA_BALANCE_MEAN,CA_BALANCE_MAX,CA_BALANCE_MIN,CA_BALANCE_LAST,CA_BALANCE_STD,TD_BALANCE_MEAN,TD_BALANCE_MAX,TD_BALANCE_LAST,TD_BALANCE_STD,CA_ACCT_MAX,TD_ACCT_MAX,MONTHS_WITH_DEPOSIT
0,4,4.411876e+05,514166.00,149274.00,514166.00,1.631847e+05,1.287097e+08,150000000.0,150000000.0,4.760661e+07,1,1,5
1,16,1.724332e+05,448415.74,20000.00,113443.48,1.889003e+05,0.000000e+00,0.0,0.0,0.000000e+00,1,0,4
2,36,4.709280e+05,1196105.20,106159.07,445701.81,3.623463e+05,0.000000e+00,0.0,0.0,0.000000e+00,1,0,9
3,43,3.473572e+05,1066767.87,154483.87,234925.81,2.943378e+05,0.000000e+00,0.0,0.0,0.000000e+00,1,0,8
4,57,5.049386e+06,29591057.90,66666.67,1303047.48,9.296444e+06,0.000000e+00,0.0,0.0,0.000000e+00,1,0,9


In [6]:
# Trend and Ratios
ca_trend = dep_nonib.groupby('CUSTOMER_NUMBER')['AVG_CA_BALANCE'].apply(compute_slope).rename('CA_BALANCE_TREND').reset_index()
td_trend = dep_nonib.groupby('CUSTOMER_NUMBER')['AVG_TD_BALANCE'].apply(compute_slope).rename('TD_BALANCE_TREND').reset_index()

dep_agg = dep_agg.merge(ca_trend, on='CUSTOMER_NUMBER', how='left')
dep_agg = dep_agg.merge(td_trend, on='CUSTOMER_NUMBER', how='left')

dep_agg['CA_TREND_DIRECTION'] = np.sign(dep_agg['CA_BALANCE_TREND']).astype('int8')
dep_agg['TD_TREND_DIRECTION'] = np.sign(dep_agg['TD_BALANCE_TREND']).astype('int8')

total_deposit = dep_agg['CA_BALANCE_MEAN'] + dep_agg['TD_BALANCE_MEAN']
dep_agg['TD_TO_TOTAL_RATIO'] = np.where(total_deposit > 0, dep_agg['TD_BALANCE_MEAN'] / total_deposit, 0.0)

dep_agg['HAS_CA'] = (dep_agg['CA_BALANCE_MAX'] > 0).astype('int8')
dep_agg['HAS_TD'] = (dep_agg['TD_BALANCE_MAX'] > 0).astype('int8')
dep_agg['DEPOSIT_DIVERSITY'] = dep_agg['HAS_CA'] + dep_agg['HAS_TD']

print(f'Deposit features final shape: {dep_agg.shape}')

Deposit features final shape: (91195, 21)


## 4. Card Features

In [7]:
df_card = read_clean_table('card_clean')
card_nonib = df_card[df_card['CUSTOMER_NUMBER'].isin(NONIB_IDS)].copy()

card_agg = card_nonib.groupby('CUSTOMER_NUMBER').agg(
    CREDITCARD_MAX   = ('COUNT_CREDITCARD', 'max'),
    DEBITCARD_MAX    = ('COUNT_DEBITCARD', 'max'),
    MONTHS_WITH_CARD = ('MONTH', 'nunique'),
).reset_index()

card_agg['HAS_CREDIT_CARD'] = (card_agg['CREDITCARD_MAX'] > 0).astype('int8')
card_agg['HAS_DEBIT_CARD']  = (card_agg['DEBITCARD_MAX'] > 0).astype('int8')
card_agg['CARD_DIVERSITY']   = card_agg['HAS_CREDIT_CARD'] + card_agg['HAS_DEBIT_CARD']

print(f'Card features shape: {card_agg.shape}')

Card features shape: (38299, 7)


## 5. Lending Features

In [8]:
df_lending = read_clean_table('lending_clean')
# Master Prep: Cap Outliers
df_lending = cap_outliers(df_lending, ['AVG_LOAN_AMOUNT'])

lend_nonib = df_lending[df_lending['CUSTOMER_NUMBER'].isin(NONIB_IDS)].copy()
lend_nonib['MONTH'] = pd.to_datetime(lend_nonib['MONTH'], errors='coerce')
lend_nonib = lend_nonib.sort_values(['CUSTOMER_NUMBER', 'MONTH'])

lend_agg = lend_nonib.groupby('CUSTOMER_NUMBER').agg(
    LOAN_AMOUNT_MEAN = ('AVG_LOAN_AMOUNT', 'mean'),
    LOAN_AMOUNT_MAX  = ('AVG_LOAN_AMOUNT', 'max'),
    LOAN_AMOUNT_LAST = ('AVG_LOAN_AMOUNT', 'last'),
    LOAN_AMOUNT_STD  = ('AVG_LOAN_AMOUNT', 'std'),
    LOAN_COUNT_MAX   = ('COUNT_OF_LOAN', 'max'),
    MONTHS_WITH_LOAN = ('MONTH', 'nunique'),
).reset_index()

lend_agg['LOAN_AMOUNT_STD'] = lend_agg['LOAN_AMOUNT_STD'].fillna(0)

loan_trend = lend_nonib.groupby('CUSTOMER_NUMBER')['AVG_LOAN_AMOUNT'].apply(compute_slope).rename('LOAN_TREND').reset_index()
lend_agg = lend_agg.merge(loan_trend, on='CUSTOMER_NUMBER', how='left')

lend_agg['HAS_LOAN'] = (lend_agg['LOAN_AMOUNT_MAX'] > 0).astype('int8')
lend_agg['LOAN_TREND_DIRECTION'] = np.sign(lend_agg['LOAN_TREND']).astype('int8')

print(f'Lending features shape: {lend_agg.shape}')

Capped AVG_LOAN_AMOUNT at [488857.00, 2828198500.00]


Lending features shape: (36274, 10)


## 7. Merge All and Cross-product Features

In [9]:
master = df_demo.copy()
master = master.merge(dep_agg, on='CUSTOMER_NUMBER', how='left')
master = master.merge(card_agg, on='CUSTOMER_NUMBER', how='left')
master = master.merge(lend_agg, on='CUSTOMER_NUMBER', how='left')

# Fill NaN
numeric_fill_cols = [
    'CA_BALANCE_MEAN', 'CA_BALANCE_MAX', 'CA_BALANCE_MIN', 'CA_BALANCE_LAST', 'CA_BALANCE_STD', 'CA_BALANCE_TREND',
    'TD_BALANCE_MEAN', 'TD_BALANCE_MAX', 'TD_BALANCE_LAST', 'TD_BALANCE_STD', 'TD_BALANCE_TREND',
    'CA_ACCT_MAX', 'TD_ACCT_MAX', 'MONTHS_WITH_DEPOSIT', 'CA_TREND_DIRECTION', 'TD_TREND_DIRECTION',
    'TD_TO_TOTAL_RATIO', 'DEPOSIT_DIVERSITY', 'CREDITCARD_MAX', 'DEBITCARD_MAX', 'MONTHS_WITH_CARD',
    'CARD_DIVERSITY', 'LOAN_AMOUNT_MEAN', 'LOAN_AMOUNT_MAX', 'LOAN_AMOUNT_LAST', 'LOAN_AMOUNT_STD',
    'LOAN_COUNT_MAX', 'MONTHS_WITH_LOAN', 'LOAN_TREND', 'LOAN_TREND_DIRECTION'
]
for col in numeric_fill_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0)

flag_cols = ['HAS_CA', 'HAS_TD', 'HAS_CREDIT_CARD', 'HAS_DEBIT_CARD', 'HAS_LOAN']
for col in flag_cols:
    if col in master.columns:
        master[col] = master[col].fillna(0).astype('int8')

# Cross-product features
master['PRODUCT_COUNT'] = (master['HAS_CA'] + master['HAS_TD'] + master['HAS_CREDIT_CARD'] + master['HAS_DEBIT_CARD'] + master['HAS_LOAN'])
master['TOTAL_DEPOSIT'] = master['CA_BALANCE_MEAN'] + master['TD_BALANCE_MEAN']
master['TOTAL_FINANCIAL_VALUE'] = master['TOTAL_DEPOSIT'] + master['LOAN_AMOUNT_MEAN']
master['NET_WORTH_PROXY'] = master['TOTAL_DEPOSIT'] - master['LOAN_AMOUNT_MEAN']
master['SAVINGS_RATE'] = np.where(master['TOTAL_FINANCIAL_VALUE'] > 0, master['TD_BALANCE_MEAN'] / master['TOTAL_FINANCIAL_VALUE'], 0.0)

master['IS_BORROWER_ONLY'] = ((master['HAS_LOAN'] == 1) & (master['TOTAL_DEPOSIT'] == 0)).astype('int8')
master['IS_SAVER_ONLY'] = ((master['HAS_TD'] == 1) & (master['HAS_LOAN'] == 0)).astype('int8')
master['IS_DORMANT'] = (master['PRODUCT_COUNT'] == 0).astype('int8')

master['MAX_ENGAGEMENT_MONTHS'] = master[['MONTHS_WITH_DEPOSIT', 'MONTHS_WITH_CARD', 'MONTHS_WITH_LOAN']].max(axis=1)

print(f'Final master shape: {master.shape}')
display(master.head())

Final master shape: (127000, 51)


,CUSTOMER_NUMBER,AGE,AGE_GROUP,TENURE_DAYS,CLIENT_SEX,SEX_F,SEX_M,CA_BALANCE_MEAN,CA_BALANCE_MAX,CA_BALANCE_MIN,CA_BALANCE_LAST,CA_BALANCE_STD,TD_BALANCE_MEAN,TD_BALANCE_MAX,TD_BALANCE_LAST,TD_BALANCE_STD,CA_ACCT_MAX,TD_ACCT_MAX,MONTHS_WITH_DEPOSIT,CA_BALANCE_TREND,TD_BALANCE_TREND,CA_TREND_DIRECTION,TD_TREND_DIRECTION,TD_TO_TOTAL_RATIO,HAS_CA,...,DEPOSIT_DIVERSITY,CREDITCARD_MAX,DEBITCARD_MAX,MONTHS_WITH_CARD,HAS_CREDIT_CARD,HAS_DEBIT_CARD,CARD_DIVERSITY,LOAN_AMOUNT_MEAN,LOAN_AMOUNT_MAX,LOAN_AMOUNT_LAST,LOAN_AMOUNT_STD,LOAN_COUNT_MAX,MONTHS_WITH_LOAN,LOAN_TREND,HAS_LOAN,LOAN_TREND_DIRECTION,PRODUCT_COUNT,TOTAL_DEPOSIT,TOTAL_FINANCIAL_VALUE,NET_WORTH_PROXY,SAVINGS_RATE,IS_BORROWER_ONLY,IS_SAVER_ONLY,IS_DORMANT,MAX_ENGAGEMENT_MONTHS
0,62117,44.035592,35-45,309,M,0,1,0.000000e+00,0.00,0.00,0.00,0.000000,3.308337e+08,370000000.0,370000000.0,1.055436e+08,0.0,2.0,11.0,0.000000,1.883536e+07,0.0,1.0,1.0,0,...,1.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0,0.0,1,3.308337e+08,3.308337e+08,3.308337e+08,1.0,0,1,0,11.0
1,900734,36.566735,35-45,6,F,1,0,0.000000e+00,0.00,0.00,0.00,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.000000,0.000000e+00,0.0,0.0,0.0,0,...,0.0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0,0.0,0,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0,0,1,0.0
2,248998,38.261465,35-45,40,M,0,1,1.032446e+06,1032446.39,1032446.39,1032446.39,0.000000,0.000000e+00,0.0,0.0,0.000000e+00,1.0,0.0,1.0,0.000000,0.000000e+00,0.0,0.0,0.0,1,...,1.0,0.0,1.0,2.0,0,1,1.0,0.0,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0,0.0,2,1.032446e+06,1.032446e+06,1.032446e+06,0.0,0,0,0,2.0
3,954356,21.311431,<25,292,M,0,1,2.133808e+04,124203.87,556.00,27857.00,40264.245056,0.000000e+00,0.0,0.0,0.000000e+00,1.0,0.0,9.0,-4765.113333,0.000000e+00,-1.0,0.0,0.0,1,...,1.0,0.0,0.0,0.0,0,0,0.0,18597224.0,33475000.0,3719448.0,1.018612e+07,1.0,9.0,-3719444.0,1,-1.0,2,2.133808e+04,1.861856e+07,-1.857589e+07,0.0,0,0,0,9.0
4,671766,34.499658,25-35,55,M,0,1,2.166667e+05,250000.00,183333.33,250000.00,47140.454436,0.000000e+00,0.0,0.0,0.000000e+00,1.0,0.0,2.0,66666.670000,0.000000e+00,1.0,0.0,0.0,1,...,1.0,0.0,0.0,0.0,0,0,0.0,270000000.0,270000000.0,270000000.0,0.000000e+00,1.0,2.0,0.0,1,0.0,2,2.166667e+05,2.702167e+08,-2.697833e+08,0.0,0,0,0,2.0


## 8. Final Output

In [10]:
save_output(master, 'nonib_master')

Saved nonib_master: shape=(127000, 51)
